# Demo 3 · From notebook to a live app 🌐
**Think of it like a restaurant.** This notebook is the **kitchen**: we test recipes here. An **app** is the **restaurant**: anyone can walk in (open a link) and order (use it), without seeing the kitchen.

We use **Streamlit**: a free tool that turns a small Python file into a web app.

**Plan:** ask AI to write the app → look at the final app file → test its brain → open it live.

## ⚙️ Setup (run once)
Click the ▶ button on the left of the cell below. When a box appears, **paste your Groq API key** and press **Enter**.

> 🔑 **No key yet?** Go to [console.groq.com/keys](https://console.groq.com/keys) → sign in with Google → **Create API Key** → copy it.
> Treat it like a password. Do not share it or put it in slides.

In [ ]:
# This cell installs the AI library and connects to Groq. You do not need to understand it.
%pip install -q groq
import re, json, html
from getpass import getpass
from groq import Groq
from IPython.display import HTML, display, Markdown

client = Groq(api_key=getpass("Paste your Groq API key and press Enter: "))
MODEL = "openai/gpt-oss-20b"   # fast, free-tier friendly. You can also try "openai/gpt-oss-120b"

def ask_ai(prompt, system="You are a friendly, helpful assistant."):
    """Send a message to the AI and get its answer back as text."""
    reply = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
    )
    return reply.choices[0].message.content

def clean_code(text):
    """AI often wraps code in ``` marks. This keeps only the code inside."""
    m = re.search(r"```(?:[a-zA-Z]*)\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def show_page(page, height=480):
    """Show a web page right here inside the notebook."""
    display(HTML(f'<iframe srcdoc="{html.escape(page)}" width="100%" height="{height}" '
                 f'style="border:1px solid #ccc;border-radius:8px"></iframe>'))

print(ask_ai("Reply with exactly: ✅ Connected! The AI is ready.", system="Follow the instruction exactly."))

## Step 1 · Ask the AI to write the whole app ✍️
Same Director move as before. This time we ask for a Streamlit app instead of a web page.

In [ ]:
app_request = """
Write a short Streamlit app in Python using the groq library.
- Title: Customer Support Triage Desk
- A password box in the sidebar for the Groq API key
- A text area for a customer email and a 'Run AI' button
- When clicked, ask the model 'openai/gpt-oss-20b' to return priority, category, a one-line summary and a polite reply
- Show the results nicely
Return only the Python code.
"""
draft_app = clean_code(ask_ai(app_request, system="You are an expert Python developer. Reply with only code."))
print(draft_app)

✅ The AI wrote a full app in seconds. In class, we did the same in **Cursor** (Ctrl+L / Cmd+L) and it saved the file for us.

## Step 2 · The final app file we deployed 📄
Below is the cleaned, tested version (`app.py`). Running this cell saves it as a file. It has 5 simple parts:

| Part | What it does | Restaurant version |
|---|---|---|
| 1. Page look | Title and short description | The sign outside |
| 2. API key box | Lets each user plug in their own key | Entry pass |
| 3. Sample emails | One-click examples | Menu of the day |
| 4. The instruction | Tells the AI exactly what to return | The recipe |
| 5. Run AI button | Sends the email, shows the answer | The waiter |

In [ ]:
%%writefile app.py
"""
Customer Support Triage Desk
A tiny AI app: paste a customer email, press Run AI, get priority, category and a reply.
Built for the MT1 "Director mindset & vibe coding" session.
"""
import json
import re

import streamlit as st
from groq import Groq

# ---------- 1. Page look ----------
st.set_page_config(page_title="Support Triage Desk", page_icon="📨", layout="centered")
st.title("📨 Customer Support Triage Desk")
st.caption("Paste a customer email. The AI reads it and tells you how urgent it is, what it is about, and how to reply.")

# ---------- 2. The API key (the AI's entry pass) ----------
with st.sidebar:
    st.header("🔑 Setup")
    try:  # a key saved in Streamlit "Secrets" is used if the box is left empty
        saved_key = st.secrets.get("GROQ_API_KEY", "")
    except Exception:
        saved_key = ""
    api_key = st.text_input(
        "Your Groq API key",
        type="password",
        value="",
        placeholder="Paste key here (starts with gsk_)",
        help="Free key from console.groq.com/keys. It stays in your browser session only.",
    ) or saved_key
    model = st.selectbox("AI model", ["openai/gpt-oss-20b", "openai/gpt-oss-120b"],
                         help="20b is faster. 120b is a bit smarter.")
    st.markdown("Get a free key: [console.groq.com/keys](https://console.groq.com/keys)")

# ---------- 3. Sample emails so anyone can try it ----------
SAMPLES = {
    "✍️ Write my own": "",
    "😡 Angry: charged twice": "Hi, I have been charged TWICE for my order #4521 and nobody is answering my calls. "
                              "I need my money back today or I will cancel my account.",
    "📦 Where is my parcel?": "Hello, I ordered a lamp last week and the tracking has not changed for 5 days. "
                             "Can you tell me when it will arrive?",
    "🙂 Small question": "Hi team, just wanted to know if you have a dark mode in the app. Thanks!",
}
choice = st.selectbox("Try a sample email", list(SAMPLES.keys()))
email = st.text_area("Customer email", value=SAMPLES[choice], height=150,
                     placeholder="Paste or type a customer email here...")

# ---------- 4. The instruction we give the AI ----------
INSTRUCTION = """You are a customer support triage assistant.
Read the customer email and answer ONLY with JSON in this exact shape:
{"priority": "High" or "Medium" or "Low",
 "category": "Billing" or "Technical" or "Delivery" or "Account" or "Other",
 "summary": "one short sentence",
 "reply": "a short, polite reply to the customer (2-3 sentences)"}"""


def triage(text: str) -> dict:
    client = Groq(api_key=api_key)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": INSTRUCTION},
                  {"role": "user", "content": text}],
        temperature=0.2,
    )
    raw = resp.choices[0].message.content or ""
    match = re.search(r"\{.*\}", raw, re.S)
    return json.loads(match.group(0) if match else raw)


# ---------- 5. The Run AI button ----------
if st.button("🤖 Run AI", type="primary", use_container_width=True):
    if not api_key:
        st.warning("Please paste your Groq API key in the left panel first. (On phone, tap the > arrow at the top left.)")
    elif not email.strip():
        st.warning("Please type or pick an email first.")
    else:
        with st.spinner("The AI is reading the email..."):
            try:
                result = triage(email)
            except Exception as e:  # keep the message friendly for beginners
                msg = str(e)
                if "401" in msg or "invalid_api_key" in msg.lower():
                    st.error("That API key did not work. Copy it again from console.groq.com/keys.")
                elif "429" in msg:
                    st.error("Too many requests right now. Wait 30 seconds and try again.")
                else:
                    st.error(f"Something went wrong: {msg}")
                st.stop()

        colour = {"High": "🔴", "Medium": "🟠", "Low": "🟢"}.get(result.get("priority"), "⚪")
        c1, c2 = st.columns(2)
        c1.metric("Priority", f"{colour} {result.get('priority', '?')}")
        c2.metric("Category", result.get("category", "?"))
        st.info(f"**Summary:** {result.get('summary', '')}")
        st.subheader("✉️ Suggested reply")
        st.write(result.get("reply", ""))

st.divider()
st.caption("Built by directing AI, not by typing every line. MT1: The Director Mindset & Vibe Coding.")

## Step 3 · Test the app's brain here first 🧪
Good chefs taste before serving. This uses the **same instruction** as the app.

In [ ]:
INSTRUCTION = """You are a customer support triage assistant.
Read the customer email and answer ONLY with JSON in this exact shape:
{"priority": "High" or "Medium" or "Low",
 "category": "Billing" or "Technical" or "Delivery" or "Account" or "Other",
 "summary": "one short sentence",
 "reply": "a short, polite reply to the customer (2-3 sentences)"}"""

email = "Hello, I cannot log in to my account since this morning. It says password expired."
answer = ask_ai(email, system=INSTRUCTION)
print(json.dumps(json.loads(re.search(r"\{.*\}", answer, re.S).group(0)), indent=2))

## Step 4 · Open the live app 🎉
👉 **Live app:** LIVE_APP_URL

1. Open the link (works on phones too).
2. Paste your Groq key in the left panel.
3. Pick a sample email and press **Run AI**.

### How it went live (4 clicks, no servers)
1. Put `app.py` and `requirements.txt` in a GitHub folder (repository).
2. Go to **share.streamlit.io** and sign in with GitHub.
3. Click **Create app** → pick the repository → main file `app.py`.
4. Click **Deploy**. In about 2 minutes you get a public link.

### Want to run it on your own laptop instead? (Cursor)
Open the folder in Cursor, open the Terminal and type:
```
pip install streamlit groq
streamlit run app.py
```
Your browser opens the app at `localhost:8501`. *Local* means it runs only on your laptop.